In [ ]:
import numpy as np

# ============================================================
# PROGRAMA 1 - MÉTODO DE BROYDEN
# Integrantes:
#  - VILLEGAS BRAVO CARLOS MAURICIO
#  - AVENDAÑO ZARZA LUIS ALBERTO
# ============================================================

# -------------------------
# Sistemas F(X) = 0
# -------------------------

def F1(X):
    x, y = X
    return np.array([
        x**2 + x*y + 2*y**2 - 5,
        5*y - 2*x*y**2 + 3
    ], dtype=float)

def J1(X):
    x, y = X
    return np.array([
        [2*x + y, x + 4*y],
        [-2*y**2, 5 - 4*x*y]
    ], dtype=float)

def F2(X):
    x, y = X
    return np.array([
        x**2 - 3*y**2 - 10,
        2*y**2 - 3*x*y + 1
    ], dtype=float)

def J2(X):
    x, y = X
    return np.array([
        [2*x, -6*y],
        [-3*y, 4*y - 3*x]
    ], dtype=float)

def F3(X):
    x, y, z = X
    return np.array([
        3*x**2 + y**2 - 8*y - 5,
        -2*x**2 - 12*x + y**2 - 3*z**2 + 10,
        -x + 2*y + 5*z
    ], dtype=float)

def J3(X):
    x, y, z = X
    return np.array([
        [6*x, 2*y - 8, 0],
        [-4*x - 12, 2*y, -6*z],
        [-1, 2, 5]
    ], dtype=float)

def F4(X):
    x, y, z = X
    return np.array([
        x**2 + y**2 - 2*z + 3,
        x + y + z - 5,
        x**2 - y**2 + z**2 - 9
    ], dtype=float)

def J4(X):
    x, y, z = X
    return np.array([
        [2*x, 2*y, -2],
        [1, 1, 1],
        [2*x, -2*y, 2*z]
    ], dtype=float)

SYSTEMS = {
    1: ("Sistema 1 (2 variables)", F1, J1, 2),
    2: ("Sistema 2 (2 variables)", F2, J2, 2),
    3: ("Sistema 3 (3 variables)", F3, J3, 3),
    4: ("Sistema 4 (3 variables)", F4, J4, 3),
}

# -------------------------
# Utilidades
# -------------------------

def norm_inf(v):
    return float(np.max(np.abs(v)))

def safe_inv(A):
    """Intenta invertir A; si falla, usa pseudo-inversa."""
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A)

def fmt_vec(v, nd=6):
    return "(" + ", ".join([f"{float(x):.{nd}f}" for x in v]) + ")"

def print_system_equations(choice):
    if choice == 1:
        print("Sistema 1:")
        print("  f1(x,y) = x^2 + x y + 2 y^2 - 5 = 0")
        print("  f2(x,y) = 5 y - 2 x y^2 + 3 = 0")
    elif choice == 2:
        print("Sistema 2:")
        print("  f1(x,y) = x^2 - 3 y^2 - 10 = 0")
        print("  f2(x,y) = 2 y^2 - 3 x y + 1 = 0")
    elif choice == 3:
        print("Sistema 3:")
        print("  f1(x,y,z) = 3x^2 + y^2 - 8y - 5 = 0")
        print("  f2(x,y,z) = -2x^2 - 12x + y^2 - 3z^2 + 10 = 0")
        print("  f3(x,y,z) = -x + 2y + 5z = 0")
    elif choice == 4:
        print("Sistema 4:")
        print("  f1(x,y,z) = x^2 + y^2 - 2z + 3 = 0")
        print("  f2(x,y,z) = x + y + z - 5 = 0")
        print("  f3(x,y,z) = x^2 - y^2 + z^2 - 9 = 0")

def read_initial_vector(n):
    while True:
        try:
            if n == 2:
                x = float(input("Ingrese x0: "))
                y = float(input("Ingrese y0: "))
                return np.array([x, y], dtype=float)
            else:
                x = float(input("Ingrese x0: "))
                y = float(input("Ingrese y0: "))
                z = float(input("Ingrese z0: "))
                return np.array([x, y, z], dtype=float)
        except ValueError:
            print("Entrada inválida. Intente de nuevo.")

# -------------------------
# Método de Broyden con Sherman–Morrison para H_k
# -------------------------

def broyden(F, J, x0, tol=1e-6, max_iter=50, verbose=True):
    """
    Broyden:
      H0 = inv(J(x0))
      s_k = -H_k F(x_k)
      x_{k+1} = x_k + s_k
      y_k = F(x_{k+1}) - F(x_k)
      H_{k+1} = H_k + ((s_k - H_k y_k) (s_k^T H_k)) / (s_k^T H_k y_k)
    """
    xk = x0.copy()
    J0 = J(xk)
    Hk = safe_inv(J0)

    if verbose:
        print("\n--- Inicio Broyden  ---")
        print("x^(0) =", fmt_vec(xk))
        print("H^(0) = inv(J(x^(0))) (o pseudo-inversa si fue necesario)\n")

        header = f"{'k':>3} | {'X^(k)':>28} | {'F(X^(k))':>32} | {'||F||_inf':>12}"
        print(header)
        print("-" * len(header))

    for k in range(max_iter + 1):
        Fk = F(xk)
        err = norm_inf(Fk)

        if verbose:
            print(f"{k:>3} | {fmt_vec(xk,6):>28} | {fmt_vec(Fk,6):>32} | {err:>12.6e}")

        if err < tol:
            return xk, err, k, True

        # Paso s_k = -H_k F_k
        sk = -Hk @ Fk
        x_next = xk + sk

        F_next = F(x_next)
        yk = F_next - Fk

        # Actualización Sherman–Morrison para H_k
        denom = float(sk.T @ (Hk @ yk))
        if abs(denom) < 1e-14:
            # Evita división por casi 0: reinicializa H con Jacobiana actual
            Hk = safe_inv(J(x_next))
        else:
            u = (sk - Hk @ yk).reshape(-1, 1)          # columna
            vT = (sk.T @ Hk).reshape(1, -1)            # renglón
            Hk = Hk + (u @ vT) / denom

        xk = x_next

    # Si salió por iteraciones
    return xk, norm_inf(F(xk)), max_iter, False

# -------------------------
# Menú principal
# -------------------------

def main():
    while True:
        print("\n==============================")
        print("PROGRAMA 1. MÉTODO DE BROYDEN")
        print("==============================")
        print("Integrantes:")
        print(" - VILLEGAS BRAVO CARLOS MAURICIO")
        print(" - AVENDAÑO ZARZA LUIS ALBERTO\n")

        print("Seleccione un sistema:")
        print("  1) Sistema 1 (2 variables)")
        print("  2) Sistema 2 (2 variables)")
        print("  3) Sistema 3 (3 variables)")
        print("  4) Sistema 4 (3 variables)")
        print("  0) Salir")

        try:
            op = int(input("Opción: "))
        except ValueError:
            print("Opción inválida.")
            continue

        if op == 0:
            print("Saliendo del programa...")
            break

        if op not in SYSTEMS:
            print("Opción inválida.")
            continue

        name, F, J, n = SYSTEMS[op]

        print("\n--------------------------------")
        print(name)
        print("--------------------------------")
        print_system_equations(op)


        x0 = read_initial_vector(n)
        try:
            tol = float(input("Ingrese tolerancia (ej. 0.0005): "))
            max_iter = int(input("Ingrese máximo de iteraciones: "))
        except ValueError:
            print("Tolerancia o iteraciones inválidas.")
            continue


        root, err, iters, ok = broyden(F, J, x0, tol=tol, max_iter=max_iter, verbose=True)

        print("\n==============================")
        print("RESULTADO")
        print("==============================")
        if ok:
            print("Converge ✅")
            print("Raíz aproximada:", fmt_vec(root, 8))
            print(f"Error (||F||_inf) en la última iteración: {err:.6e}")
            print(f"Iteraciones realizadas: {iters}")
        else:
            print("No converge ❌")
            print("Última aproximación:", fmt_vec(root, 8))
            print(f"Error (||F||_inf) final: {err:.6e}")
            print(f"Iteraciones máximas alcanzadas: {iters}")

        # Menú de otro cálculo
        again = input("\n¿Desea probar otros datos iniciales? (S/N): ").strip().lower()
        if again == "s":
            continue

        other = input("¿Desea elegir otro sistema? (S/N): ").strip().lower()
        if other == "s":
            continue

        back = input("¿Desea regresar al menú principal? (S/N): ").strip().lower()
        if back == "s":
            continue

        print("Saliendo del programa...")
        break

# Ejecutar
if __name__ == "__main__":
    main()


PROGRAMA 1. MÉTODO DE BROYDEN
Integrantes:
 - VILLEGAS BRAVO CARLOS MAURICIO
 - AVENDAÑO ZARZA LUIS ALBERTO

Seleccione un sistema:
  1) Sistema 1 (2 variables)
  2) Sistema 2 (2 variables)
  3) Sistema 3 (3 variables)
  4) Sistema 4 (3 variables)
  0) Salir
Opción: 4

--------------------------------
Sistema 4 (3 variables)
--------------------------------
Sistema 4:
  f1(x,y,z) = x^2 + y^2 - 2z + 3 = 0
  f2(x,y,z) = x + y + z - 5 = 0
  f3(x,y,z) = x^2 - y^2 + z^2 - 9 = 0
Ingrese x0: 5
Ingrese y0: 8
Ingrese z0: -2
Ingrese tolerancia (ej. 0.0005): 0.0004
Ingrese máximo de iteraciones: 16

--- Inicio Broyden  ---
x^(0) = (5.000000, 8.000000, -2.000000)
H^(0) = inv(J(x^(0))) (o pseudo-inversa si fue necesario)

  k |                        X^(k) |                         F(X^(k)) |    ||F||_inf
------------------------------------------------------------------------------------
  0 | (5.000000, 8.000000, -2.000000) | (96.000000, 6.000000, -44.000000) | 9.600000e+01
  1 | (2.636364, 3.57